# S2-08 `@hobo/provider-cloudflare` draft POC — HOBO-623

The Cloudflare provider draft proves three things: it **covers the initial capability ports**
(`web/service`, `kv`, `objectStore`) through published equivalence classes, it **publishes a
canonical 4-document well-known capability manifest** (deterministic bytes, SVID federation,
never static secrets), and it **plans hello-bindle** with typed Cloudflare option blocks and
WASI binds. All demonstrations below call the package's real modules.

In [1]:
import { strict as assert } from "node:assert";
import {
  createProviderCloudflareDraft, equivalenceForPort, INITIAL_CAPABILITY_PORTS,
  MANIFEST_SCHEMA, parsePublishedManifest, publishCapabilityManifest,
} from "./src/index.ts";

const provider = createProviderCloudflareDraft();
console.log(`provider: ${provider.id}`);
for (const port of INITIAL_CAPABILITY_PORTS) {
  const mapping = equivalenceForPort(port);
  assert.ok(provider.coversInitialPort(port));
  console.log(`port ${port} -> ${mapping.resourceClass} (${mapping.providerService})`);
}

// The published well-known set parses clean and republishes byte-identically.
const documents = provider.publishedManifest;
const parsed = parsePublishedManifest(documents);
assert.ok(parsed.ok);
assert.equal(parsed.index.manifestSchema, MANIFEST_SCHEMA);
assert.deepEqual(publishCapabilityManifest({ colos: ["WNAM", "ENAM", "WEUR"] }), documents);
console.log(`manifest parse ok=${parsed.ok} schema=${MANIFEST_SCHEMA} deterministic=true`);

provider: @hobo/provider-cloudflare
port web/service -> cloudflare:workers-script (Workers)
port kv -> cloudflare:kv-namespace (KV)
port objectStore -> cloudflare:r2-bucket (R2)
manifest parse ok=true schema=hobo.capability-manifest/v1 deterministic=true


## Fail-closed parsing and the hello-bindle plan

The manifest parse boundary rejects any document set that leaks static secrets; the
hello-bindle plan wires compute to `kv`/`objectStore` via WASI-style binds, and typed option
blocks merge caller overrides over provider defaults.

In [2]:
import { strict as assert } from "node:assert";
import { cloudflare, createProviderCloudflareDraft, parsePublishedManifest } from "./src/index.ts";

const provider = createProviderCloudflareDraft();
const documents = provider.publishedManifest;

// Tampered auth document (static secret) is rejected at the parse boundary.
const tamperedAuth = JSON.parse(documents["/.well-known/capability-manifest/auth.json"]);
tamperedAuth.staticKey = "sk-oops";
const rejected = parsePublishedManifest({
  ...documents,
  "/.well-known/capability-manifest/auth.json": JSON.stringify(tamperedAuth),
});
assert.equal(rejected.ok, false);
console.log(`tampered manifest rejected: ${rejected.errors.join("; ")}`);

// hello-bindle plan: three resources, compute bound to the other two ports.
const plan = provider.helloBindlePlan;
console.log(`plan ${plan.id}: resources=${plan.resources.map((r) => `${r.id}:${r.type}`).join(",")}`);
const service = plan.resources.find((resource) => resource.id === "service");
for (const bind of service.binds) console.log(`bind: ${bind.import} -> ${bind.providerResourceId} (${bind.capability})`);

// Typed option blocks: overrides win, defaults survive.
const worker = cloudflare.with.worker({ cpuMs: 50 });
assert.equal(worker.options.cpuMs, 50);
assert.equal(worker.options.memoryMb, 128);
console.log(`option block: port=${worker.port} cpuMs=${worker.options.cpuMs} memoryMb=${worker.options.memoryMb}`);
console.log("CONTRACT HELD: cloudflare provider publishes a canonical, secret-free capability manifest covering the initial ports");

tampered manifest rejected: auth must not publish static secrets
plan hello-bindle-cloudflare: resources=service:compute,assets:objectStore,sessions:kv
bind: wasi:blobstore/blobstore -> assets (objectStore)
bind: hobo:kv/store -> sessions (kv)
option block: port=compute cpuMs=50 memoryMb=128
CONTRACT HELD: cloudflare provider publishes a canonical, secret-free capability manifest covering the initial ports
